# svrecon benchmark

3960 SVs (22 types x 20 each x 3 size classes x 3 modes = 1320 per mode) simulated with
insilicoSV, then scored by svrecon under three CIGAR-validation modes, on a positive and a
negative callset, one pair per size class.

| arm | callset | assembly | expected |
|---|---|---|---|
| positive | seed 0 | seed 0 | 440/440 hits per size class |
| negative | seed 1 | seed 0 | 0/440 hits per size class |

Hits in the negative arm are false positives of that mode.

Configs live in `insilicoSV/` and `svrecon/` beside this notebook, generated by
`insilicoSV/generate_configs.py` and `svrecon/generate_configs.py`; this notebook only runs them.

**Prerequisites:** run from this notebook's own directory (`workflows/`), with `insilicosv`
and `svrecon` pip-installed. All paths below are relative to it.

In [ ]:
%%bash
set -euo pipefail
command -v insilicosv >/dev/null || { echo 'insilicosv not on PATH'; exit 1; }
command -v svrecon >/dev/null || { echo 'svrecon not on PATH'; exit 1; }
test -f insilicoSV/small-0/insilicoSV.yaml || { echo 'run me from the workflows directory'; exit 1; }
echo ok

## 1. Download inputs

hg38 chr21 and its RepeatMasker intervals. The rmsk table is genome-wide (~180 MB),
piped down to chr21 rows. Skips anything already present.

In [ ]:
%%bash
set -euo pipefail
mkdir -p data

[ -f data/chr21.fa ] || \
  curl -fSL https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr21.fa.gz \
  | gunzip -c > data/chr21.fa

# rmsk.txt columns: 6 genoName, 7 genoStart, 8 genoEnd, 12 repClass
[ -f data/chr21.rmsk.bed ] || \
  curl -fSL https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/rmsk.txt.gz \
  | gunzip -c \
  | awk -F'\t' -v OFS='\t' '$6=="chr21" {print $6, $7, $8, $12}' \
  | sort -k1,1 -k2,2n > data/chr21.rmsk.bed

ls -lh data
wc -l < data/chr21.rmsk.bed

## 2. Simulate

Writes `sim.vcf` / `sim.hapA.fa` beside each config: 3 size classes x 2 seeds. ~10 min.

In [ ]:
%%bash
set -euo pipefail

for size in small medium large; do
  for seed in 0 1; do
    (cd "insilicoSV/${size}-${seed}" && insilicosv -c insilicoSV.yaml)
  done
done

## 3. Score

18 runs: 3 size classes x 2 arms x 3 modes, 440 SVs each. Writes `svrecon.log` beside each
config. ~1.5-2 hr.

In [ ]:
%%bash
set -euo pipefail

for size in small medium large; do
  for arm in "$size" "${size}_neg"; do
    for mode in similarity similarity--no-large-errors similarity--windowered-errors; do
      (cd "svrecon/${arm}/${mode}" && svrecon --config svrecon.yaml)
    done
  done
done

### Results

In [ ]:
import json, re
from pathlib import Path

SIZES = ['small', 'medium', 'large']
MODES = ['similarity', 'similarity--no-large-errors', 'similarity--windowered-errors']

def outcomes(log):
    """SVID -> outcome, from the per-SV JSON each verbose log line ends with. Pretty-printed
    (indent=2) so it spans several lines; raw_decode anchored right at the '{' after the tab is
    indent-agnostic, unlike matching the whole blob with a single-line regex."""
    text = open(log).read()
    decoder = json.JSONDecoder()
    out = {}
    for m in re.finditer(r'\t\{\n', text):
        rec, _ = decoder.raw_decode(text, m.start() + 1)
        out[rec['svid']] = rec['outcome']
    return out

rows = []
for mode in MODES:
    for arm, suffix in [('positive', ''), ('negative', '_neg')]:
        per_size, hits, inconclusive, total = [], 0, 0, 0
        for size in SIZES:
            out = outcomes(Path('svrecon') / f'{size}{suffix}' / mode / 'svrecon.log')
            hit = sum(v == 'hit' for v in out.values())
            per_size.append(f'{hit}/{len(out)}')
            hits += hit
            inconclusive += sum(v == 'inconclusive' for v in out.values())
            total += len(out)
        conclusive = total - inconclusive
        precision = hits / conclusive if conclusive else float('nan')
        rows.append([arm, mode, *per_size, f'{hits}/{total}', f'{inconclusive}/{total}',
                     f'{precision:.2f}'])

hdr = ['arm', 'mode', *SIZES, 'total', 'inconclusive', 'precision']
w = [max(len(str(r[i])) for r in [hdr] + rows) for i in range(len(hdr))]
line = lambda r: '  '.join(str(c).ljust(w[i]) for i, c in enumerate(r))
print(line(hdr))
print('  '.join('-' * x for x in w))
for r in rows:
    print(line(r))
print('\nnegative arm: hits are false positives, lower is better')
print('precision excludes inconclusive calls from the denominator: hits / (total - inconclusive)')